# ✈️ Airline Sentiment Analysis — BERT Training
**Магистрант:** Комаров Николай Александрович

**GPU:** Google Colab Free (T4)

Этот ноутбук обучает DistilBERT для 3 задач:
1. Sentiment (positive/negative/neutral)
2. Category (baggage/booking/delay/...)
3. Criticality (low/medium/high)

## 1. Setup

In [ ]:
!pip install -q transformers torch scikit-learn pandas numpy tqdm
import torch
print(f'PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Upload Project
Загрузите zip-архив проекта или клонируйте репозиторий:

In [ ]:
# Option A: Clone from GitHub (замените URL)
# !git clone https://github.com/YOUR_USERNAME/airline-sentiment.git
# %cd airline-sentiment

# Option B: Upload zip and extract
from google.colab import files
import zipfile, os

if not os.path.exists('airline-sentiment'):
    print('Upload airline-sentiment.zip:')
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith('.zip'):
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall('.')
            print(f'Extracted {fname}')

%cd airline-sentiment
!ls -la

## 3. Generate Data & Train Baseline

In [ ]:
!python scripts/generate_data.py
!python scripts/train.py --baseline-only

## 4. Train DistilBERT (GPU)

In [ ]:
import sys
sys.path.insert(0, '.')
from src.data import prepare_dataset, split_dataset
from src.bert_model import BERTTrainer, print_test_report

# Prepare data
df = prepare_dataset()
train_df, val_df, test_df = split_dataset(df)

print(f'Dataset: {len(train_df)} train, {len(val_df)} val, {len(test_df)} test')
print(f'Device: {"cuda" if torch.cuda.is_available() else "cpu"}')

In [ ]:
# Train BERT — ~15 min on T4 GPU
trainer = BERTTrainer()
history = trainer.train(train_df, val_df, epochs=5)

## 5. Evaluate on Test Set

In [ ]:
print_test_report(trainer, test_df)

## 6. Test Predictions

In [ ]:
examples = [
    'My flight was delayed 6 hours and nobody helped us. Terrible service!',
    'Amazing crew, comfortable seats, arrived early. Best airline!',
    'Average flight. Nothing special but got me there on time.',
    'Lost my luggage and customer service hung up on me. I want compensation!',
    'Check-in was smooth and boarding was fast. Decent experience.',
]

for text in examples:
    result = trainer.predict_single(text)
    print(f'\nText: {text}')
    for task, r in result.items():
        print(f'  {task}: {r["label"]} ({r["confidence"]:.2f})')

## 7. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend()

f1_scores = [m['sentiment']['f1_macro'] for m in history['val_metrics']]
ax2.plot(f1_scores, label='Sentiment F1-macro', marker='o')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('F1-macro')
ax2.set_title('Validation Sentiment F1'); ax2.legend()

plt.tight_layout()
plt.savefig('reports/training_curves.png', dpi=150)
plt.show()

## 8. Confusion Matrix

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from src.config import SENTIMENT_LABELS

# Get predictions
loader = trainer._make_loader(test_df)
trainer.model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in loader:
        ids = batch['input_ids'].to(trainer.device)
        mask = batch['attention_mask'].to(trainer.device)
        out = trainer.model(ids, mask)
        all_preds.extend(out['sentiment'].argmax(1).cpu().numpy())
        all_labels.extend(batch['sentiment'].numpy())

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=SENTIMENT_LABELS).plot(ax=ax, cmap='Blues')
plt.title('Sentiment Confusion Matrix (BERT)')
plt.tight_layout()
plt.savefig('reports/confusion_matrix.png', dpi=150)
plt.show()

## 9. Save & Download Model

In [ ]:
# Save final metrics
import json

test_metrics, _ = trainer.evaluate(test_df)
report = {'bert': {k: {kk: round(vv, 4) for kk, vv in v.items()} for k, v in test_metrics.items()}}

with open('reports/metrics_bert.json', 'w') as f:
    json.dump(report, f, indent=2)

print('Metrics saved!')
print(json.dumps(report, indent=2))

In [ ]:
# Download trained model
!zip -r trained_model.zip models/bert/
from google.colab import files
files.download('trained_model.zip')
files.download('reports/metrics_bert.json')
files.download('reports/training_curves.png')
files.download('reports/confusion_matrix.png')